# Gatra on Colab T4

Runtime → Change runtime type → GPU (T4).
Checkpoints go to Google Drive so a disconnect does not wipe the run.

In [ ]:
!nvidia-smi -L
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U uv
!git clone --depth 1 https://github.com/ManifoldSystems/gatra.git
%cd gatra
!uv sync --extra data

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!uv run gatra-prepare --lang id --limit 20000 --out data/culturax-id.jsonl

In [ ]:
from pathlib import Path
drive_ckpts = Path('/content/drive/MyDrive/gatra/checkpoints')
drive_ckpts.mkdir(parents=True, exist_ok=True)
print(drive_ckpts)

Train Gatra-1M first as a CUDA smoke. Then Gatra-10M with a time budget.

In [ ]:
!uv run gatra-train --config configs/gatra-1m.toml --device cuda --out-dir /content/drive/MyDrive/gatra/checkpoints --max-time 20m

In [ ]:
!uv run gatra-train --config configs/gatra-10m.toml --device cuda --out-dir /content/drive/MyDrive/gatra/checkpoints --max-time 50m

If Colab disconnects, resume from Drive:

In [ ]:
!uv run gatra-continue --config configs/gatra-10m.toml --device cuda --out-dir /content/drive/MyDrive/gatra/checkpoints --which latest --extra-iters 2000 --max-time 50m

In [ ]:
!uv run gatra-generate --checkpoint /content/drive/MyDrive/gatra/checkpoints/gatra-10m/best.pt --device cuda --prompt 'Pagi itu'